In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [2]:
df = pd.read_csv("../dataset/adult_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (45194, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
X = df.drop("income", axis=1)

y = (
    df["income"]
    .astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
    .eq(">50K")
    .astype(int)
)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nIncome distribution:")
print(y.value_counts())

Features shape: (45194, 14)
Target shape: (45194,)

Income distribution:
income
0    33988
1    11206
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (36155, 14)
Testing data: (9039, 14)


In [5]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_columns = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numerical",
            "passthrough",
            numerical_columns
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=8,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Baseline Decision Tree model trained successfully!")

C:\Users\Admin\AppData\Local\Temp\ipykernel_11312\2395606339.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


Baseline Decision Tree model trained successfully!


In [6]:
y_pred = model.predict(X_test)

print("Predictions completed!")
print("First 10 predictions:", y_pred[:10])

Predictions completed!
First 10 predictions: [0 0 0 0 0 0 1 1 0 0]


In [7]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("BASELINE MODEL PERFORMANCE")
print("--------------------------")

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

BASELINE MODEL PERFORMANCE
--------------------------
Accuracy: 0.8491
Precision: 0.7739
Recall: 0.5529
F1 Score: 0.645

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.90      6798
           1       0.77      0.55      0.64      2241

    accuracy                           0.85      9039
   macro avg       0.82      0.75      0.77      9039
weighted avg       0.84      0.85      0.84      9039



In [8]:
results = pd.DataFrame({
    "sex": X_test["sex"].values,
    "actual": y_test.values,
    "predicted": y_pred
})

gender_metrics = results.groupby("sex").apply(
    lambda group: pd.Series({
        "Selection Rate": group["predicted"].mean(),
        "True Positive Rate": (
            group.loc[group["actual"] == 1, "predicted"].mean()
        )
    }),
    include_groups=False
)

print("FAIRNESS RESULTS BY GENDER")
print("--------------------------")
print(gender_metrics)

FAIRNESS RESULTS BY GENDER
--------------------------
        Selection Rate  True Positive Rate
sex                                       
Female        0.065077            0.430678
Male          0.230996            0.574658


In [9]:
demographic_parity_difference = (
    gender_metrics["Selection Rate"].max()
    - gender_metrics["Selection Rate"].min()
)

equal_opportunity_difference = (
    gender_metrics["True Positive Rate"].max()
    - gender_metrics["True Positive Rate"].min()
)

print("Demographic Parity Difference:",
      round(demographic_parity_difference, 4))

print("Equal Opportunity Difference:",
      round(equal_opportunity_difference, 4))

Demographic Parity Difference: 0.1659
Equal Opportunity Difference: 0.144
